In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [3]:
df = pd.read_csv(r"data\data.csv")

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
df.shape


(7043, 21)

In [7]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## Churn distribution

In [ ]:
import seaborn as sns

plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x="Churn", order=["No", "Yes"], palette="Set2")

total = len(df)
for p in ax.patches:
    count = int(p.get_height())
    pct = 100 * count / total
    ax.annotate(
        f"{pct:.1f}%",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=10,
        xytext=(0, 3),
        textcoords="offset points",
    )

ax.set_title("Churn distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Numeric feature distributions

In [ ]:
df_num = df.copy()
df_num["TotalCharges"] = pd.to_numeric(df_num["TotalCharges"], errors="coerce")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(
    data=df_num,
    x="tenure",
    hue="Churn",
    bins=30,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[0],
)
axes[0].set_title("Tenure distribution by churn")

sns.histplot(
    data=df_num,
    x="MonthlyCharges",
    hue="Churn",
    bins=30,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[1],
)
axes[1].set_title("MonthlyCharges distribution by churn")

sns.histplot(
    data=df_num,
    x="TotalCharges",
    hue="Churn",
    bins=30,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[2],
)
axes[2].set_title("TotalCharges distribution by churn")

plt.tight_layout()
plt.show()

## Categorical churn rates

In [ ]:
def churn_rate_by(column: str) -> pd.DataFrame:
    out = (
        df.assign(Churn01=(df["Churn"].str.strip().str.lower() == "yes").astype(int))
        .groupby(column, as_index=False)["Churn01"]
        .mean()
        .rename(columns={"Churn01": "churn_rate"})
    )
    out["churn_rate_pct"] = out["churn_rate"] * 100
    return out.sort_values("churn_rate_pct", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, col in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    rates = churn_rate_by(col)
    sns.barplot(data=rates, x="churn_rate_pct", y=col, ax=ax, palette="Blues_r")
    ax.set_title(f"Churn rate by {col}")
    ax.set_xlabel("Churn rate (%)")
    ax.set_ylabel("")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%", padding=3)

plt.tight_layout()
plt.show()

## Correlation heatmap

In [ ]:
df_corr = df.copy()
df_corr["TotalCharges"] = pd.to_numeric(df_corr["TotalCharges"], errors="coerce")

yes_no_cols = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

for col in yes_no_cols:
    if col in df_corr.columns:
        df_corr[col] = df_corr[col].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})

# Target
df_corr["Churn"] = df_corr["Churn"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})

corr_cols = ["Churn", "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"] + yes_no_cols
corr = df_corr[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation heatmap (numeric + binary Yes/No)")
plt.tight_layout()
plt.show()

## Key takeaways

- **Contract type is a major churn driver**: month-to-month customers churn at a much higher rate than customers on 1–2 year contracts.
- **Internet service type and payment method matter**: Fiber optic customers and electronic-check payments tend to show higher churn rates than other categories.
- **Tenure and charges separate churners from non-churners**: churners are concentrated at lower tenure, and their monthly/total charges distributions differ from retained customers.

In [9]:
df.duplicated().sum()



np.int64(0)